In [44]:
from rdflib import Graph, Namespace, RDF
from rdflib.namespace import OWL
from langchain.embeddings import HuggingFaceEmbeddings
import numpy as np
from pathlib import Path


In [45]:
# ── Load graph ────────────────────────────────────────────────────────────────
DANCE  = Namespace("http://example.org/dance/")
SCHEMA = Namespace("http://schema.org/")

g = Graph()
g.parse("./graphs/KG_base_dance.ttl", format="turtle")
print(f"Loaded {len(g)} triples")


Loaded 5911 triples


In [46]:
# ── Embedding model (same as link_KG.ipynb) ───────────────────────────────────
model_name = "BAAI/bge-large-en-v1.5"
embedding_model = HuggingFaceEmbeddings(model_name=model_name, model_kwargs={"device": "cpu"})
print("Embedding model ready.")


Embedding model ready.


In [47]:

def get_nodes(class_uri):
    nodes = []
    for uri in set(g.subjects(RDF.type, class_uri)):
        label = g.value(uri, SCHEMA.name)
        if label:
            nodes.append((uri, str(label)))
    return nodes


In [48]:
# ── Helper: merge duplicates found by embedding similarity ───────────────────
def merge_similar(class_uri, threshold=0.97, verbose=True):
    nodes = get_nodes(class_uri)
    if len(nodes) < 2:
        print(f"  {class_uri.split('/')[-1]}: fewer than 2 nodes, skipping.")
        return 0

    uris   = [n[0] for n in nodes]
    labels = [n[1] for n in nodes]

    # Embed all labels at once
    vectors = np.array(embedding_model.embed_documents(labels), dtype="float32")
    # L2-normalise for cosine similarity via dot product
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    vectors = vectors / np.where(norms == 0, 1, norms)

    merges = 0
    merged_into: dict = {}

    for i in range(len(uris)):
        for j in range(i + 1, len(uris)):
            sim = float(np.dot(vectors[i], vectors[j]))
            if sim < threshold:
                continue

            if labels[i] <= labels[j]:
                canonical, duplicate = uris[i], uris[j]
                canon_label, dup_label = labels[i], labels[j]
            else:
                canonical, duplicate = uris[j], uris[i]
                canon_label, dup_label = labels[j], labels[i]

            while duplicate in merged_into:
                duplicate = merged_into[duplicate]
            if duplicate == canonical:
                continue

            merged_into[duplicate] = canonical
            if verbose:
                print(f"  MERGE ({sim:.3f})  '{dup_label}'  →  '{canon_label}'")

            for s, p, _ in list(g.triples((None, None, duplicate))):
                g.remove((s, p, duplicate))
                g.add((s, p, canonical))
            for _, p, o in list(g.triples((duplicate, None, None))):
                g.remove((duplicate, p, o))
                g.add((canonical, p, o))

            merges += 1

    print(f"  {class_uri.split('/')[-1]}: {merges} merge(s) from {len(uris)} nodes.")
    return merges


In [52]:
# ── Run merging for every multi-valued property class ────────────────────────
# Threshold: 0.97 = very similar (near-duplicates / paraphrases).
# Lower it (e.g. 0.92) to merge broader synonyms.
THRESHOLD = 0.85

classes_to_merge = [
    DANCE.Instrument,
    DANCE.MusicGenre,
    DANCE.TimePeriod,
    DANCE.Origin,
    #DANCE.Practitioner,
    #DANCE.HealthBenefit,
]

total_merges = 0
for cls in classes_to_merge:
    print(f"\n── {cls.split('/')[-1]} ──")
    total_merges += merge_similar(cls, threshold=THRESHOLD)

print(f"\nTotal merges: {total_merges}")
print(f"Triples after merging: {len(g)}")



── Instrument ──
  Instrument: 0 merge(s) from 98 nodes.

── MusicGenre ──
  MusicGenre: 0 merge(s) from 98 nodes.

── TimePeriod ──
  TimePeriod: 0 merge(s) from 30 nodes.

── Origin ──
  Origin: 0 merge(s) from 36 nodes.

Total merges: 0
Triples after merging: 5824


In [54]:
# ── Save the cleaned graph ────────────────────────────────────────────────────
out_path = Path("../graphs/KG_base_dance_merged.ttl")
g.serialize(destination=str(out_path), format="turtle")
print(f"Saved merged graph to {out_path}")


Saved merged graph to ../graphs/KG_base_dance_merged.ttl
